# Arabic Text Preprocessing for RAG Systems

This notebook is a hands-on lab covering how to preprocess **Arabic text** before feeding it
into a Retrieval-Augmented Generation (RAG) pipeline (chunking → embedding → vector store → retrieval).

Arabic brings a distinct set of challenges that most English-first NLP tutorials never touch.
If you skip them, your embeddings will be noisy, your retriever will miss obvious matches, and your
chunks will break mid-word. This notebook walks through each challenge with runnable code, then
assembles everything into a single `preprocess()` function you can drop into a real pipeline.

## Challenges covered
1. **Diacritics (Tashkeel)** — vowel marks that create sparse, inconsistent tokens
2. **Letter-shape normalization** — Alef variants (ا أ إ آ), Teh Marbuta (ة) vs Heh (ه), Alef Maksura (ى) vs Yeh (ي)
3. **Tatweel / Kashida (ـ)** — elongation characters used for justification
4. **Character elongation** — repeated letters from informal writing ("جدااااا")
5. **Digits** — Arabic-Indic (٠١٢) vs Persian (۰۱۲) vs Western (012) numerals
6. **Punctuation** — Arabic-specific punctuation (، ؛ ؟) and mixed punctuation
7. **Stopwords** — high-frequency function words that add noise to retrieval
8. **Stemming / Light stemming** — trade-offs between root extraction and meaning preservation
9. **Tokenization** — word and sentence segmentation without Latin-style capitalization cues
10. **Code-switching** — mixed Arabic/English text (very common in tech, academic, and business text)
11. **Dialectal Arabic vs. Modern Standard Arabic (MSA)** — informal variants that don't normalize cleanly
12. **RTL (right-to-left) rendering** considerations for debugging/display
13. **Chunking strategy** for RAG that respects Arabic sentence/word boundaries
14. **Putting it all together**: a full preprocessing pipeline + a mini retrieval demo

> **Note on tooling**: this notebook uses lightweight, pip-installable libraries (`pyarabic`, `nltk`)
> that work offline. In production, you would likely pair this preprocessing with an
> Arabic-aware embedding model (e.g. `AraBERT`, `camel-bert`, or a multilingual model such as
> `multilingual-e5` / `paraphrase-multilingual-mpnet`) via `sentence-transformers`. The preprocessing
> logic here is model-agnostic and applies regardless of which embedding model you choose.


## 0. Setup

In [2]:
!pip install pyarabic arabic-reshaper python-bidi nltk scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.4/126.4 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.2/296.2 kB 4.6 MB/s eta 0:00:00


In [3]:
# If running for the first time, install dependencies:
# !pip install pyarabic arabic-reshaper python-bidi nltk scikit-learn

import re
import unicodedata
import nltk

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

import pyarabic.araby as araby
from nltk.corpus import stopwords
from nltk.stem.isri import ISRIStemmer

print("Setup complete.")


Setup complete.


## 1. A messy, realistic sample corpus

RAG systems usually ingest real-world text: scraped web pages, PDFs, forum posts, or user-generated
content. Real Arabic text is rarely "clean" — it mixes diacritics, dialect, elongation, digits, and
even English terms. Let's use a small corpus that reflects this.


In [4]:
raw_documents = [
    # MSA with full diacritics (e.g. from a religious/classical/educational source)
    "اَلْعَرَبِيَّةُ لُغَةٌ جَمِيلَةٌ وَغَنِيَّةٌ بِالْمُفْرَدَاتِ وَالتَّرَاكِيبِ.",

    # informal social-media style: elongation, no diacritics, mixed punctuation
    "الله الله عليك يا بطل!!! الأداء كان رهيييييب جداااا 🔥🔥",

    # mixed Arabic/English (code-switching), common in tech & business writing
    "يستخدم نظام RAG تقنية retrieval augmented generation لتحسين إجابات الـ LLM باستخدام قاعدة المعرفة الداخلية.",

    # numerals in Arabic-Indic form + inconsistent Alef/Teh Marbuta spellings
    "بلغت الإيرادات في سنة ٢٠٢٤ حوالي ٩٥٪، بزيادة قدرها ١٠ نقاط عن العام الماضي.",

    # dialectal Arabic (Egyptian) mixed with MSA — common in customer support transcripts
    "معلش يا باشا، الموضوع ده هيتحل بسرعة إن شاء الله، بس محتاجين شوية وقت كمان.",

    # inconsistent hamza / alef forms across the same corpus
    "إحسان أحمد ذهب إلى امتحان في آخر الأسبوع، ثم عاد الى البيت مبكرا.",
]

for i, d in enumerate(raw_documents, 1):
    print(f"[{i}] {d}")


[1] اَلْعَرَبِيَّةُ لُغَةٌ جَمِيلَةٌ وَغَنِيَّةٌ بِالْمُفْرَدَاتِ وَالتَّرَاكِيبِ.
[2] الله الله عليك يا بطل!!! الأداء كان رهيييييب جداااا 🔥🔥
[3] يستخدم نظام RAG تقنية retrieval augmented generation لتحسين إجابات الـ LLM باستخدام قاعدة المعرفة الداخلية.
[4] بلغت الإيرادات في سنة ٢٠٢٤ حوالي ٩٥٪، بزيادة قدرها ١٠ نقاط عن العام الماضي.
[5] معلش يا باشا، الموضوع ده هيتحل بسرعة إن شاء الله، بس محتاجين شوية وقت كمان.
[6] إحسان أحمد ذهب إلى امتحان في آخر الأسبوع، ثم عاد الى البيت مبكرا.


## 2. Challenge 1 — Diacritics (Tashkeel)

Arabic diacritics (`َ ُ ِ ّ ْ ً ٌ ٍ`) indicate short vowels and gemination. Most written Arabic
(news, web pages, forums) **omits them entirely**, but formal/classical/religious texts often
include them fully. If your corpus mixes both styles, the *same word* will produce different
tokens/embeddings depending on whether it's diacritized — hurting retrieval recall.

**Rule of thumb for RAG:** strip diacritics during preprocessing for retrieval consistency, but
consider keeping the original (diacritized) text as the stored/returned chunk if diacritics carry
meaningful disambiguation for your domain (e.g. Quranic text, poetry, pronunciation-sensitive apps).


In [5]:
sample = raw_documents[0]
print("Before:", sample)
print("After :", araby.strip_tashkeel(sample))

# strip_tashkeel removes harakat but keeps shadda separate handling if needed;
# strip_diacritics is a stronger pass that also normalizes some remaining marks.
print("Strong :", araby.strip_diacritics(sample))


Before: اَلْعَرَبِيَّةُ لُغَةٌ جَمِيلَةٌ وَغَنِيَّةٌ بِالْمُفْرَدَاتِ وَالتَّرَاكِيبِ.
After : العربية لغة جميلة وغنية بالمفردات والتراكيب.
Strong : العربية لغة جميلة وغنية بالمفردات والتراكيب.


## 3. Challenge 2 — Letter-shape normalization

Arabic has several letters with variant forms that are **semantically identical** in most
contexts but are visually/encoding-wise distinct:

| Variants | Normalized to | Notes |
|---|---|---|
| `أ إ آ ٱ` (Alef with hamza/madda) | `ا` (bare Alef) | Almost always safe for retrieval |
| `ى` (Alef Maksura) | `ي` (Yeh) | Common spelling confusion, esp. at word end |
| `ة` (Teh Marbuta) | `ه` (Heh) | Debatable — teh marbuta marks feminine nouns; normalize with care |
| `ئ ؤ` (Hamza on Yeh/Waw) | sometimes kept, sometimes stripped to `ي`/`و` | Depends on domain |

Normalizing these variants means a user searching "احمد" will also match documents containing
"أحمد", "إحمد" written by a different author.


In [6]:
sample2 = raw_documents[5]
print("Before:", sample2)

normalized = araby.normalize_alef(sample2)   # أ إ آ -> ا
print("Alef normalized:", normalized)

normalized = araby.normalize_hamza(normalized)  # normalizes hamza forms
print("Hamza normalized:", normalized)

# Teh marbuta -> Heh (use with caution; can blur feminine/masculine distinction)
teh_example = "مدرسة كبيرة وحديقة جميلة"
print("\nTeh Marbuta example:")
print("Before:", teh_example)
print("After :", araby.normalize_teh(teh_example))

# Alef Maksura vs Yeh
maksura_example = "على الفتى أن يسعى الى مستقبل أفضل"
print("\nAlef Maksura example:")
print("Before:", maksura_example)
print("After :", araby.normalize_alef_maksura(maksura_example) if hasattr(araby, "normalize_alef_maksura") else maksura_example.replace("ى", "ي"))


Before: إحسان أحمد ذهب إلى امتحان في آخر الأسبوع، ثم عاد الى البيت مبكرا.
Alef normalized: احسان احمد ذهب الا امتحان في اخر الاسبوع، ثم عاد الا البيت مبكرا.
Hamza normalized: احسان احمد ذهب الا امتحان في اخر الاسبوع، ثم عاد الا البيت مبكرا.

Teh Marbuta example:
Before: مدرسة كبيرة وحديقة جميلة
After : مدرسه كبيره وحديقه جميله

Alef Maksura example:
Before: على الفتى أن يسعى الى مستقبل أفضل
After : علي الفتي أن يسعي الي مستقبل أفضل


## 4. Challenge 3 — Tatweel / Kashida removal

`ـ` (tatweel) is a decorative elongation character used to visually justify text
(e.g. `مـــرحبا`). It carries **no linguistic meaning** and must be stripped, or it will
fragment otherwise-identical words into unique, unmatched tokens.


In [7]:
tatweel_example = "مـــرحـــبا بكــم في نظـــام البحث"
print("Before:", tatweel_example)
print("After :", araby.strip_tatweel(tatweel_example))


Before: مـــرحـــبا بكــم في نظـــام البحث
After : مرحبا بكم في نظام البحث


## 5. Challenge 4 — Character elongation (informal writing)

Distinct from tatweel, **repeated letters** are an informal way of adding emphasis in social
media / chat text, e.g. "رهيييييب" (roughly: "amaaaazing"). A regex collapsing 3+ repeated
characters down to one handles this without corrupting legitimately doubled letters (shadda-like
doubling is rare in undiacritized text, but keep the threshold at 3 to be safe with words that
naturally have 2 repeated letters, e.g. "مجموعة").


In [8]:
def remove_elongation(text: str) -> str:
    """Collapse 3+ repeated characters into a single character."""
    return re.sub(r"(.)\1{2,}", r"\1", text)

elong_example = raw_documents[1]
print("Before:", elong_example)
print("After :", remove_elongation(elong_example))


Before: الله الله عليك يا بطل!!! الأداء كان رهيييييب جداااا 🔥🔥
After : الله الله عليك يا بطل! الأداء كان رهيب جدا 🔥🔥


## 6. Challenge 5 — Digits (Arabic-Indic, Persian, Western)

Arabic text may use **Western digits** (0-9), **Arabic-Indic digits** (٠١٢٣٤٥٦٧٨٩, common in
Gulf/Levant text), or **Persian/Urdu digits** (۰۱۲۳۴۵۶۷۸۹, sometimes seen in text influenced by
Persian typing tools). For consistent retrieval and downstream numeric parsing, normalize all to
Western digits.


In [9]:
AR_INDIC_TO_WEST = str.maketrans("٠١٢٣٤٥٦٧٨٩", "0123456789")
FA_TO_WEST       = str.maketrans("۰۱۲۳۴۵۶۷۸۹", "0123456789")

def normalize_digits(text: str) -> str:
    return text.translate(AR_INDIC_TO_WEST).translate(FA_TO_WEST)

digits_example = raw_documents[3]
print("Before:", digits_example)
print("After :", normalize_digits(digits_example))


Before: بلغت الإيرادات في سنة ٢٠٢٤ حوالي ٩٥٪، بزيادة قدرها ١٠ نقاط عن العام الماضي.
After : بلغت الإيرادات في سنة 2024 حوالي 95٪، بزيادة قدرها 10 نقاط عن العام الماضي.


## 7. Challenge 6 — Punctuation normalization

Arabic uses its own punctuation marks that are visually mirrored versions of Latin ones:
`،` (comma), `؛` (semicolon), `؟` (question mark). Real-world text often **mixes** Arabic and
Latin punctuation inconsistently. Normalize them to a single convention, and strip characters that
add no semantic value for retrieval (while keeping sentence-ending punctuation for sentence
segmentation, done in the next section).


In [10]:
ARABIC_PUNCT_MAP = {
    "،": ",",
    "؛": ";",
    "؟": "?",
    "٪": "%",
    "”": '"',
    "“": '"',
    "’": "'",
    "‘": "'",
}

def normalize_punctuation(text: str) -> str:
    for ar, la in ARABIC_PUNCT_MAP.items():
        text = text.replace(ar, la)
    return text

punct_example = "هل هذا صحيح؟ نعم، بالتأكيد؛ فهو موثق بنسبة ٩٩٪."
print("Before:", punct_example)
print("After :", normalize_punctuation(punct_example))


Before: هل هذا صحيح؟ نعم، بالتأكيد؛ فهو موثق بنسبة ٩٩٪.
After : هل هذا صحيح? نعم, بالتأكيد; فهو موثق بنسبة ٩٩%.


## 8. Challenge 7 — Tokenization & sentence segmentation

Arabic has no capitalization to signal proper nouns or sentence starts, and word boundaries can
be ambiguous around attached clitics (e.g. `و` "and", `ال` "the", `ب` "by/with" are prefixed
directly onto the following word: `والكتاب` = `و` + `الكتاب` = "and the book"). Off-the-shelf
whitespace tokenizers mostly work for word-level splitting since Arabic *is* space-delimited at
the word level, but generic (English-tuned) sentence tokenizers often fail on Arabic punctuation.
`pyarabic.araby.sentence_tokenize` and a small custom regex handle this better.


In [11]:
text_block = "هذا مثال بسيط. وهذه جملة أخرى! هل يعمل الترميز بشكل صحيح؟ نعم، يبدو ذلك."

print("Word tokens:")
print(araby.tokenize(text_block))

print("\nSentence tokens:")
for s in araby.sentence_tokenize(text_block):
    print("-", s)


Word tokens:
['هذا', 'مثال', 'بسيط', '.', 'وهذه', 'جملة', 'أخرى', '!', 'هل', 'يعمل', 'الترميز', 'بشكل', 'صحيح', '؟', 'نعم', '،', 'يبدو', 'ذلك', '.']

Sentence tokens:
- هذا مثال بسيط.
- وهذه جملة أخرى! هل يعمل الترميز بشكل صحيح؟
- نعم،
- يبدو ذلك.


## 9. Challenge 8 — Stopwords

Just like English "the", "is", "and" — Arabic has a large set of high-frequency function words
(`في`, `من`, `على`, `الذي`, `هذا`, ...) that usually add noise rather than signal for dense/sparse
retrieval matching. NLTK ships a solid Arabic stopword list (750+ words). Whether to remove
stopwords for RAG depends on your retriever:

- **Sparse retrieval (BM25/TF-IDF)** → removing stopwords usually helps.
- **Dense embeddings (transformer-based)** → modern embedding models are trained on natural text
  and often perform *better* if you leave stopwords in, since removing them can break sentence
  structure/semantics the model relies on. Test empirically on your domain.


In [12]:
arabic_stopwords = set(stopwords.words("arabic"))
print(f"Loaded {len(arabic_stopwords)} Arabic stopwords. Examples:", list(arabic_stopwords)[:10])

def remove_stopwords(text: str, stopword_set=arabic_stopwords) -> str:
    tokens = araby.tokenize(text)
    filtered = [t for t in tokens if t not in stopword_set]
    return " ".join(filtered)

sw_example = "الذكاء الاصطناعي هو أحد أهم المجالات التي تطورت في السنوات الأخيرة"
print("\nBefore:", sw_example)
print("After :", remove_stopwords(sw_example))


Loaded 701 Arabic stopwords. Examples: ['نفس', 'يا', 'ؤ', 'خ', 'عليه', 'ر', 'إليكَ', 'هَذا', 'إلا', 'إليكم']

Before: الذكاء الاصطناعي هو أحد أهم المجالات التي تطورت في السنوات الأخيرة
After : الذكاء الاصطناعي أهم المجالات تطورت السنوات الأخيرة


## 10. Challenge 9 — Stemming vs. light stemming

Arabic is a **root-and-pattern (templatic) language**: words are built from a 3-letter root
(e.g. `ك-ت-ب` "writing-related") combined with patterns/affixes to produce related but
semantically distinct words (`كتب` "wrote/books", `مكتبة` "library", `كاتب` "writer",
`مكتوب` "written/destiny").

- **Aggressive root stemming** (e.g. ISRI stemmer) reduces all of these to the same root — great
  for maximizing recall in sparse search, but it can **over-collapse distinct meanings**, hurting
  precision.
- **Light stemming** only strips common prefixes/suffixes (`ال`, `و`, `ة`, `ين`, `ات`...) without
  reducing to the root — usually the better default for RAG, since it normalizes inflectional
  variants without destroying lexical meaning.

Below we compare both on the same words so you can see the trade-off directly.


In [13]:
stemmer = ISRIStemmer()

words = ["الكاتب", "مكتبة", "كتاب", "يكتبون", "الكاتبون", "مكتوب"]

print(f"{'word':<12}{'ISRI root stem':<18}{'light stem':<15}")
for w in words:
    root_stem = stemmer.stem(w)
    light = araby.strip_tashkeel(araby.strip_tatweel(w))
    # simple light-stemming: strip definite article + common suffixes
    light = re.sub(r"^(ال|و|ف|ب|ك|ل)", "", light)
    light = re.sub(r"(ون|ين|ات|ة|ه|ها|هم)$", "", light)
    print(f"{w:<12}{root_stem:<18}{light:<15}")

print("\nNotice how ISRI collapses 'مكتبة' (library) and 'يكتبون' (they write) toward the")
print("same root 'كتب', losing the distinction that a RAG retriever likely needs to keep.")


word        ISRI root stem    light stem     
الكاتب      كتب               كاتب           
مكتبة       كتب               مكتب           
كتاب        كتب               تاب            
يكتبون      كتب               يكتب           
الكاتبون    كتب               كاتب           
مكتوب       كتب               مكتوب          

Notice how ISRI collapses 'مكتبة' (library) and 'يكتبون' (they write) toward the
same root 'كتب', losing the distinction that a RAG retriever likely needs to keep.


## 11. Challenge 10 — Code-switching (mixed Arabic/English)

Technical, academic, and business Arabic text very commonly embeds English terms
(`RAG`, `LLM`, `API`) without transliteration. A preprocessing pipeline must avoid corrupting the
Latin spans (e.g. don't run Arabic-only regexes that accidentally strip Latin letters) and should
optionally flag/segment mixed-script documents so you can route them to the right
tokenizer/embedding model, or make sure your chosen embedding model handles both scripts well
(most multilingual embedding models do, but Arabic-only models like AraBERT may treat English
spans as unknown tokens).


In [14]:
def detect_script_mix(text: str) -> dict:
    arabic_chars = len(re.findall(r"[\u0600-\u06FF]", text))
    latin_chars  = len(re.findall(r"[A-Za-z]", text))
    total = max(arabic_chars + latin_chars, 1)
    return {
        "arabic_chars": arabic_chars,
        "latin_chars": latin_chars,
        "arabic_ratio": round(arabic_chars / total, 2),
        "latin_ratio": round(latin_chars / total, 2),
        "is_mixed": arabic_chars > 0 and latin_chars > 0,
    }

mixed_example = raw_documents[2]
print(mixed_example)
print(detect_script_mix(mixed_example))


يستخدم نظام RAG تقنية retrieval augmented generation لتحسين إجابات الـ LLM باستخدام قاعدة المعرفة الداخلية.
{'arabic_chars': 58, 'latin_chars': 34, 'arabic_ratio': 0.63, 'latin_ratio': 0.37, 'is_mixed': True}


## 12. Challenge 11 — Dialectal Arabic vs. MSA

Modern Standard Arabic (MSA, `فصحى`) is used in formal writing, but user-generated content
(support tickets, reviews, social media, transcribed calls) is frequently written in a regional
**dialect** (Egyptian, Gulf, Levantine, Maghrebi, ...). Dialects:

- Have different vocabulary for common words (`عايز`/`عاوز` vs. MSA `أريد` = "I want")
- Don't have a single standardized spelling, so the *same* dialectal word appears spelled multiple
  ways across documents
- Are not reliably "fixed" by the normalization rules above, since the differences are lexical,
  not just orthographic

For RAG systems ingesting dialectal content, the practical options are:
1. **Detect dialect vs. MSA** and route to a dialect-aware embedding model if available.
2. **Keep dialectal text as-is** and rely on a robust multilingual embedding model — most modern
   multilingual embedding models have seen enough Arabic dialect data to handle this reasonably.
3. **Build/extend a normalization dictionary** mapping common dialectal terms to their MSA
   equivalent (only worth it for known, high-frequency domain terms).

We don't have a lightweight offline dialect identifier bundled here, but here's a simple
illustration of option 3: a tiny domain-specific normalization dictionary.


In [15]:
# A tiny illustrative dialect -> MSA normalization dictionary.
# In production this would be built from your domain's actual dialectal vocabulary,
# or replaced by a proper dialect-ID model (e.g. from CAMeL Tools).
DIALECT_TO_MSA = {
    "عايز": "أريد",
    "عاوز": "أريد",
    "ازيك": "كيف حالك",
    "معلش": "لا بأس",
    "ده": "هذا",
    "كده": "هكذا",
    "بس": "لكن",
}

def normalize_dialect(text: str, mapping=DIALECT_TO_MSA) -> str:
    tokens = text.split()
    return " ".join(mapping.get(tok, tok) for tok in tokens)

dialect_example = raw_documents[4]
print("Before:", dialect_example)
print("After :", normalize_dialect(dialect_example))


Before: معلش يا باشا، الموضوع ده هيتحل بسرعة إن شاء الله، بس محتاجين شوية وقت كمان.
After : لا بأس يا باشا، الموضوع هذا هيتحل بسرعة إن شاء الله، لكن محتاجين شوية وقت كمان.


## 13. A note on RTL rendering

Arabic is written right-to-left, and mixed Arabic/English/number text can render in a confusing
order in some terminals/notebook cells. This doesn't affect the actual string data your model
sees (Unicode strings are stored logically, not visually), but if you need to **display** Arabic
text correctly reshaped for visual debugging in environments that don't handle bidi text well,
use `arabic-reshaper` + `python-bidi`. This is purely a display concern.


In [16]:
import arabic_reshaper
from bidi.algorithm import get_display

display_example = "نظام RAG يدعم اللغة العربية بكفاءة"
reshaped = arabic_reshaper.reshape(display_example)
bidi_text = get_display(reshaped)
print("Logical string   :", display_example)
print("Visual (for plots):", bidi_text)
print("\n(Note: in Jupyter, the plain string above usually already renders correctly;")
print(" reshaping matters mainly for image/plot rendering libraries like matplotlib.)")


Logical string   : نظام RAG يدعم اللغة العربية بكفاءة
Visual (for plots): ﺓﺀﺎﻔﻜﺑ ﺔﻴﺑﺮﻌﻟﺍ ﺔﻐﻠﻟﺍ ﻢﻋﺪﻳ RAG ﻡﺎﻈﻧ

(Note: in Jupyter, the plain string above usually already renders correctly;
 reshaping matters mainly for image/plot rendering libraries like matplotlib.)


## 14. Putting it together — a full preprocessing pipeline

Now let's assemble everything into a single configurable function. For RAG, you typically want
**two variants** of each chunk:
- a **normalized version** used to build the retrieval index (embeddings/BM25) — aggressively
  cleaned for consistent matching
- the **original (or lightly cleaned) version** stored as the actual chunk text — so the text
  shown to the LLM/user preserves readability and meaning


In [19]:
def preprocess_arabic(
    text: str,
    strip_diacritics: bool = True,
    normalize_letters: bool = True,
    strip_tatweel_chars: bool = True,
    fix_elongation: bool = True,
    normalize_nums: bool = True,
    normalize_punct: bool = True,
    remove_stop: bool = False,   # default False: keep for dense embeddings, see discussion above
    dialect_map: dict = None,
) -> str:
    """Full Arabic normalization pipeline for RAG indexing.

    Returns a cleaned string suitable for building a retrieval index (BM25 or embeddings).
    Keep the original text separately if you want to return human-readable chunks to the LLM.
    """
    out = text

    if strip_diacritics:
        out = araby.strip_diacritics(out)

    if strip_tatweel_chars:
        out = araby.strip_tatweel(out)

    if fix_elongation:
        out = re.sub(r"(.)\1{2,}", r"\1", out)

    if normalize_letters:
        out = araby.normalize_alef(out)
        out = araby.normalize_hamza(out)

    if normalize_nums:
        out = out.translate(AR_INDIC_TO_WEST).translate(FA_TO_WEST)

    if normalize_punct:
        out = normalize_punctuation(out)

    if dialect_map:
        out = normalize_dialect(out, dialect_map)

    if remove_stop:
        out = remove_stopwords(out)

    # collapse extra whitespace produced by the transformations above
    out = re.sub(r"\s+", " ", out).strip()
    return out


print("Pipeline test on the full sample corpus:\n")
for i, d in enumerate(raw_documents, 1):
    cleaned = preprocess_arabic(d, dialect_map=DIALECT_TO_MSA)
    print(f"[{i}] RAW    : {d}")
    print(f"    CLEANED: {cleaned}\n")


Pipeline test on the full sample corpus:

[1] RAW    : اَلْعَرَبِيَّةُ لُغَةٌ جَمِيلَةٌ وَغَنِيَّةٌ بِالْمُفْرَدَاتِ وَالتَّرَاكِيبِ.
    CLEANED: العربية لغة جميلة وغنية بالمفردات والتراكيب.

[2] RAW    : الله الله عليك يا بطل!!! الأداء كان رهيييييب جداااا 🔥🔥
    CLEANED: الله الله عليك يا بطل! الاداء كان رهيب جدا 🔥🔥

[3] RAW    : يستخدم نظام RAG تقنية retrieval augmented generation لتحسين إجابات الـ LLM باستخدام قاعدة المعرفة الداخلية.
    CLEANED: يستخدم نظام RAG تقنية retrieval augmented generation لتحسين اجابات ال LLM باستخدام قاعدة المعرفة الداخلية.

[4] RAW    : بلغت الإيرادات في سنة ٢٠٢٤ حوالي ٩٥٪، بزيادة قدرها ١٠ نقاط عن العام الماضي.
    CLEANED: بلغت الايرادات في سنة 2024 حوالي 95%, بزيادة قدرها 10 نقاط عن العام الماضي.

[5] RAW    : معلش يا باشا، الموضوع ده هيتحل بسرعة إن شاء الله، بس محتاجين شوية وقت كمان.
    CLEANED: لا بأس يا باشا, الموضوع هذا هيتحل بسرعة ان شاء الله, لكن محتاجين شوية وقت كمان.

[6] RAW    : إحسان أحمد ذهب إلى امتحان في آخر الأسبوع، ثم عاد الى البيت مبك

## 15. Chunking strategy for RAG

Chunking Arabic text has two extra considerations beyond the usual "don't split mid-sentence":

1. **Never split inside a word.** Arabic clitics (`و`, `ال`, `ب`, `ف`, `ل`, `س`) are attached
   directly to the following word with no space — but that's still one *token*, so whitespace-based
   splitting is safe. The risk is with naive **character-count** chunking (e.g. "first 500
   characters") which can cut a word in half. Always chunk on sentence or word boundaries, never
   raw character counts.
2. **Use an Arabic-aware sentence tokenizer** (shown in section 8) rather than a generic
   period-based splitter, since Arabic punctuation and abbreviation patterns differ from English.

Below is a simple sentence-aware chunker that packs sentences into chunks up to a target word
budget (a stand-in for a token budget — swap in your embedding model's actual tokenizer for
production use).


In [17]:
def chunk_text(text: str, max_words: int = 40) -> list:
    """Pack sentences into chunks without exceeding max_words, without splitting mid-sentence."""
    sentences = araby.sentence_tokenize(text)
    chunks, current, current_len = [], [], 0

    for sent in sentences:
        n_words = len(araby.tokenize(sent))
        if current and current_len + n_words > max_words:
            chunks.append(" ".join(current))
            current, current_len = [], 0
        current.append(sent)
        current_len += n_words

    if current:
        chunks.append(" ".join(current))
    return chunks


long_document = " ".join(raw_documents)  # simulate one long document made of several sentences
chunks = chunk_text(long_document, max_words=15)

for i, c in enumerate(chunks, 1):
    print(f"Chunk {i} ({len(araby.tokenize(c))} words): {c}\n")


Chunk 1 (7 words): اَلْعَرَبِيَّةُ لُغَةٌ جَمِيلَةٌ وَغَنِيَّةٌ بِالْمُفْرَدَاتِ وَالتَّرَاكِيبِ.

Chunk 2 (27 words): الله الله عليك يا بطل!!! الأداء كان رهيييييب جداااا 🔥🔥 يستخدم نظام RAG تقنية retrieval augmented generation لتحسين إجابات الـ LLM باستخدام قاعدة المعرفة الداخلية.

Chunk 3 (8 words): بلغت الإيرادات في سنة ٢٠٢٤ حوالي ٩٥٪،

Chunk 4 (12 words): بزيادة قدرها ١٠ نقاط عن العام الماضي. معلش يا باشا،

Chunk 5 (14 words): الموضوع ده هيتحل بسرعة إن شاء الله، بس محتاجين شوية وقت كمان.

Chunk 6 (15 words): إحسان أحمد ذهب إلى امتحان في آخر الأسبوع، ثم عاد الى البيت مبكرا.



## 16. Mini retrieval demo

To close the loop, let's build a tiny TF-IDF-based retriever over our cleaned chunks and run a
query. This stands in for a dense embedding + vector store setup (e.g. FAISS/Chroma +
`multilingual-e5` embeddings) — the preprocessing you apply before indexing works the same
regardless of which retrieval backend you use.

> In production, swap `TfidfVectorizer` for embeddings from an Arabic-aware or multilingual model
> via `sentence-transformers`, and swap cosine-similarity-over-matrix for a proper vector DB.


In [20]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Build the index from the *normalized* text (for matching), but keep original text to return to the user.
normalized_chunks = [preprocess_arabic(c, dialect_map=DIALECT_TO_MSA) for c in raw_documents]
original_chunks = raw_documents

vectorizer = TfidfVectorizer(tokenizer=araby.tokenize, lowercase=False)
doc_matrix = vectorizer.fit_transform(normalized_chunks)

def retrieve(query: str, top_k: int = 2):
    cleaned_query = preprocess_arabic(query, dialect_map=DIALECT_TO_MSA)
    query_vec = vectorizer.transform([cleaned_query])
    scores = cosine_similarity(query_vec, doc_matrix)[0]
    ranked = sorted(enumerate(scores), key=lambda x: x[1], reverse=True)[:top_k]
    return [(original_chunks[i], score) for i, score in ranked if score > 0]


query = "ما هي نسبة زيادة الإيرادات؟"  # "What is the revenue increase percentage?"
print("Query:", query, "\n")
for doc, score in retrieve(query):
    print(f"score={score:.3f} | {doc}")


Query: ما هي نسبة زيادة الإيرادات؟ 

score=0.259 | بلغت الإيرادات في سنة ٢٠٢٤ حوالي ٩٥٪، بزيادة قدرها ١٠ نقاط عن العام الماضي.


## 17. Summary & best practices checklist

When building the ingestion pipeline for an Arabic (or Arabic + English) RAG system:

- [ ] **Strip diacritics** for indexing consistency (keep original for display if needed)
- [ ] **Normalize letter variants**: Alef forms → `ا`, Hamza forms, and decide deliberately on
  Teh Marbuta / Alef Maksura normalization based on your domain
- [ ] **Strip tatweel** (`ـ`) — always safe to remove
- [ ] **Collapse elongated characters** from informal text
- [ ] **Normalize digits** (Arabic-Indic/Persian → Western) if you need numeric parsing/matching
- [ ] **Normalize punctuation** for consistent sentence segmentation
- [ ] **Choose your stopword policy** based on retriever type (sparse vs. dense)
- [ ] **Prefer light stemming over aggressive root stemming** unless you specifically need
  maximum recall over precision
- [ ] **Use Arabic-aware sentence/word tokenizers**, not generic English-tuned ones
- [ ] **Detect and handle code-switching** (mixed Arabic/English) so technical terms aren't
  corrupted or dropped
- [ ] **Account for dialectal variation** if your source data includes user-generated content
- [ ] **Chunk on sentence/word boundaries**, never raw character counts
- [ ] **Pick an embedding model that actually supports Arabic well** (AraBERT/CAMeLBERT for
  Arabic-only content, or a strong multilingual model like `multilingual-e5-large` for mixed
  Arabic/English corpora) — no amount of preprocessing fixes a poor embedding model.

With these steps in place, your Arabic RAG pipeline will retrieve much more reliably than if you
ran raw text through an English-oriented pipeline.
